# 03 · nnU-Net segmentation (one fold per session)

S0 is the 4-channel nnU-Net (dataset 501). S2 is prior-conditioned (dataset 502). Checkpoints are written to Drive, so an interrupted fold resumes with `--c`.

In [ ]:
# --- setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['BMF_DRIVE'] = '/content/drive/MyDrive/bmets'                       # change if you use another folder
os.environ['BMF_DATA'] = os.environ['BMF_DRIVE'] + '/UCSD-BMETS-Longitudinal'   # Aspera download target (uploaded to Drive)
os.makedirs(os.environ['BMF_DRIVE'], exist_ok=True)
REPO = '/content/claude_projects'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rittik047/claude_projects.git $REPO
%cd /content/claude_projects/bmets-forecast
!pip -q install -e ".[dl,extra]"
CFG = 'configs/colab.yaml'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!bmf seg-export --config $CFG
os.environ['nnUNet_raw'] = os.environ['BMF_DRIVE'] + '/cache/nnunet_raw'
os.environ['nnUNet_preprocessed'] = os.environ['BMF_DRIVE'] + '/cache/nnunet_preprocessed'
os.environ['nnUNet_results'] = os.environ['BMF_DRIVE'] + '/cache/nnunet_results'

In [ ]:
DATASET, FOLD = 501, 0
import glob, shutil
name = os.path.basename(glob.glob(os.environ['nnUNet_raw'] + f'/Dataset{DATASET}_*')[0])
if not os.path.exists(os.environ['nnUNet_preprocessed'] + '/' + name):
    !nnUNetv2_plan_and_preprocess -d $DATASET -c 3d_fullres --verify_dataset_integrity
shutil.copy(os.environ['nnUNet_raw'] + f'/{name}/splits_final.json', os.environ['nnUNet_preprocessed'] + f'/{name}/splits_final.json')
!nnUNetv2_train $DATASET 3d_fullres $FOLD -tr nnUNetTrainer --npz --c

After all 5 folds of a dataset: gather the out-of-fold predictions and score them lesion-wise.

In [ ]:
M = 'S0' if DATASET == 501 else 'S2'
out = os.environ['BMF_DRIVE'] + f'/cache/seg_pred/{M}'; os.makedirs(out, exist_ok=True)
!cp $nnUNet_results/{name}/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_*/validation/*.nii.gz {out}/
!bmf seg-eval --config $CFG --pred-dir {out} --method {M}